In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from seqeval.metrics import classification_report, f1_score

In [ ]:
def load_conll_data(filepath):
    sentences = []
    labels = []
    sentence = []
    label = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip() == "":
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
            else:
                word, pos, chunk, ner = line.strip().split()
                sentence.append(word)
                label.append(ner)

    if sentence:
        sentences.append(sentence)
        labels.append(label)

    return sentences, labels

In [ ]:
train_file = r"D:\Hackathon\Atharva Dataset Train.txt"
val_file   = r"D:\Hackathon\Atharva Dataset Valid.txt"
test_file  = r"D:\Hackathon\Atharva Dataset Test.txt"

train_sentences, train_labels = load_conll_data(train_file)
val_sentences, val_labels     = load_conll_data(val_file)
test_sentences, test_labels   = load_conll_data(test_file)

In [ ]:
label_list = sorted(list(set(l for labels in train_labels for l in labels)))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

num_labels = len(label_list)

In [ ]:
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(label2id[label[word_idx]])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
train_dataset = Dataset.from_dict({"tokens": train_sentences, "ner_tags": train_labels})
val_dataset   = Dataset.from_dict({"tokens": val_sentences, "ner_tags": val_labels})
test_dataset  = Dataset.from_dict({"tokens": test_sentences, "ner_tags": test_labels})

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
val_dataset   = val_dataset.map(tokenize_and_align_labels, batched=True)
test_dataset  = test_dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50
)

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        true_labels.append([id2label[l] for l in lab if l != -100])
        true_predictions.append(
            [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        )

    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate(test_dataset)
print("Final Test F1:", metrics["eval_f1"])

In [18]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_sentence(sentence):
    words = sentence.split()

    encoding = tokenizer(
        words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True
    )

    word_ids = encoding.word_ids()

    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    preds = predictions[0].cpu().numpy()

    results = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue

        if word_idx != previous_word_idx:
            label = id2label[preds[idx]]
            results.append((words[word_idx], label))

        previous_word_idx = word_idx

    return results

# ==============================
# User Input Loop
# ==============================

while True:
    sentence = input("\nEnter a sentence (or type 'exit' to quit): ")
    
    if sentence.lower() == "exit":
        break

    results = predict_sentence(sentence)
    
    print("\nPredictions:")
    for word, tag in results:
        print(f"{word:15} -> {tag}")


Predictions:
Reliance        -> I-ORG
Retail          -> I-ORG
is              -> O
a               -> O
big             -> O
company.        -> O

Predictions:
Atharva         -> B-PER
singh           -> I-PER
works           -> O
at              -> O
FN              -> B-ORG
MathLogic.      -> I-ORG
